In [ ]:
import os
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import jieba
from torch.utils.data import Dataset, DataLoader
import time
import random
import tqdm

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def NormalizeEnglishString(s):
    s=s.lower().strip()
    s=re.sub(r'([.,!?;:\'"\-()%$])',r' \1',s)
    s=re.sub(r'[^a-zA-Z0-9.,!?;:\'"\-()%$]+',r' ',s)
    return s

In [ ]:
with open('../data/zh-en.txt','r',encoding='utf-8') as f:
    lines=f.readlines()

In [ ]:
my_pairs=[]
for line in lines:
    zh,en=line.strip().split('\t')
    en=NormalizeEnglishString(en)
    my_pairs.append((zh,en))

In [ ]:
print('zh-en pairs:',len(my_pairs))

In [ ]:
PAD_token=0
SOS_token=1
EOS_token=2
UNK_token=3
chinese_word2index={'PAD':PAD_token,'SOS':SOS_token,'EOS':EOS_token,'UNK':UNK_token}
english_word2index={'PAD':PAD_token,'SOS':SOS_token,'EOS':EOS_token,'UNK':UNK_token}

In [ ]:
if not os.path.exists('../models/word2index_and_word_count.pt'):
    chinese_word_count={}
    for zh,_ in my_pairs:
        for word in jieba.lcut(zh):
            chinese_word_count[word] = chinese_word_count.get(word, 0) + 1
    english_word_count={}
    for _,en in my_pairs:
        for word in en.split():
            english_word_count[word] = english_word_count.get(word, 0) + 1

    chinese_word_counter=sorted(chinese_word_count.items(),key=lambda x:x[1],reverse=True)
    english_word_counter=sorted(english_word_count.items(),key=lambda x:x[1],reverse=True)

    for word,i in chinese_word_counter:
        if i>=3 and word not in chinese_word2index:
            chinese_word2index[word]=len(chinese_word2index)
    for word,i in english_word_counter:
        if i>=2 and word not in english_word2index:
            english_word2index[word]=len(english_word2index)

    chinese_word_count=len(chinese_word2index)
    english_word_count=len(english_word2index)

    torch.save({'chinese_word2index':chinese_word2index,
                'english_word2index':english_word2index,
                'chinese_word_count':chinese_word_count,
                'english_word_count':english_word_count},
               '../models/word2index_and_word_count.pt')

else:
    word2index_and_word_count = torch.load('../models/word2index_and_word_count.pt')
    chinese_word2index = word2index_and_word_count['chinese_word2index']
    english_word2index = word2index_and_word_count['english_word2index']
    chinese_word_count = word2index_and_word_count['chinese_word_count']
    english_word_count = word2index_and_word_count['english_word_count']

In [ ]:
class My_Dataset(Dataset):
    def __init__(self, pairs, chinese_word2index, english_word2index):
        self.pairs = pairs
        self.chinese_word2index = chinese_word2index
        self.english_word2index = english_word2index

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        zh, en = self.pairs[idx]
        x = [self.chinese_word2index.get(word, UNK_token) for word in jieba.lcut(zh)] + [EOS_token]
        y = [self.english_word2index.get(word, UNK_token) for word in en.split()] + [EOS_token]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [ ]:
class C2E_Translator_Encoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super(C2E_Translator_Encoder,self).__init__()
        self.input_size=input_size
        self.hidden_size=hidden_size
        self.num_layers=num_layers
        self.embedding=nn.Embedding(input_size,hidden_size)
        self.gru=nn.GRU(hidden_size,hidden_size,num_layers,batch_first=True)

    def forward(self,x,hidden,length):
        embedded=self.embedding(x)
        output,hidden=self.gru(embedded,hidden)
        lengths=torch.tensor(length, dtype=torch.long, device=output.device)
        batch_idx=torch.arange(output.size(0), device=output.device)
        real_last=output[batch_idx,lengths-1,:]
        hidden=real_last.unsqueeze(0)
        return output,hidden

    def init_hidden(self,batch_size):
        return torch.zeros(self.num_layers,batch_size,self.hidden_size,device=device)

In [ ]:
class C2E_Translator_Decoder_Attention(nn.Module):
    def __init__(self,hidden_size,output_size,num_layers=1):
        super(C2E_Translator_Decoder_Attention,self).__init__()
        self.hidden_size=hidden_size
        self.output_size=output_size
        self.num_layers=num_layers
        self.embedding=nn.Embedding(output_size,hidden_size)
        self.query_combine=nn.Linear(hidden_size*2,hidden_size)
        self.attn_combine=nn.Linear(hidden_size*2,hidden_size)
        self.gru=nn.GRU(hidden_size,hidden_size,num_layers,batch_first=True)
        self.out=nn.Linear(hidden_size,output_size)
        self.dropout=nn.Dropout(0.1)

    def forward(self,input,hidden,encoder_outputs,mask):
        embedded=self.embedding(input)
        embedded=self.dropout(embedded)
        query=self.query_combine(torch.cat((embedded,hidden[-1].unsqueeze(1)),dim=-1))
        permuted_encoder_outputs=encoder_outputs.permute(0,2,1)
        attn_weights=torch.bmm(query,permuted_encoder_outputs)
        attn_weights=attn_weights.masked_fill(mask.unsqueeze(1)==0,float('-inf'))
        attn_weights=F.softmax(attn_weights,dim=-1)
        attn_applied=torch.bmm(attn_weights,encoder_outputs)
        attn_combined=self.attn_combine(torch.cat((query,attn_applied),dim=-1))
        attn_combined=F.relu(attn_combined)
        output,hidden=self.gru(attn_combined,hidden)
        output=self.out(output)
        return output,hidden,attn_weights

    def init_hidden(self,batch_size):
        return torch.zeros(self.num_layers,batch_size,self.hidden_size,device=device)

In [ ]:
encoder=C2E_Translator_Encoder(input_size=chinese_word_count,hidden_size=256,num_layers=1).to(device)
decoder=C2E_Translator_Decoder_Attention(hidden_size=256,output_size=english_word_count,num_layers=1).to(device)

In [ ]:
def collate_fn(batch):
    x_batch, y_batch = zip(*batch)
    x_lengths = [len(x) for x in x_batch]
    y_lengths = [len(y) for y in y_batch]
    max_x_length = max(x_lengths)
    max_y_length = max(y_lengths)

    padded_x_batch = torch.full((len(batch), max_x_length), PAD_token, dtype=torch.long)
    padded_y_batch = torch.full((len(batch), max_y_length), PAD_token, dtype=torch.long)

    for i, (x, y) in enumerate(batch):
        padded_x_batch[i, :len(x)] = x
        padded_y_batch[i, :len(y)] = y

    return padded_x_batch.to(device), padded_y_batch.to(device), x_lengths, y_lengths

In [ ]:
batch_size=256
my_dataloader=DataLoader(My_Dataset(my_pairs,chinese_word2index,english_word2index),batch_size=batch_size,shuffle=True,collate_fn=collate_fn)
criterion=nn.CrossEntropyLoss(ignore_index=PAD_token)
optimizer_encoder=torch.optim.Adam(encoder.parameters(),lr=0.0001)
optimizer_decoder=torch.optim.Adam(decoder.parameters(),lr=0.0001)

In [ ]:
epochs=10
interval=1000
teacher_forcing_ratio=0.5

In [ ]:
for epoch in range(epochs):
    encoder.train()
    decoder.train()
    start_time=time.time()
    iter_num=0
    total_loss=0.0
    interval_loss=0.0
    for x,y,x_length,y_length in tqdm.tqdm(my_dataloader):
        iter_num+=1
        x=x.to(device)
        y=y.to(device)
        mask = (x != PAD_token).to(device)
        batch_size=x.size(0)
        encoder_hidden=encoder.init_hidden(batch_size)
        encoder_outputs,encoder_hidden=encoder(x,encoder_hidden,x_length)
        decoder_input=torch.tensor([[SOS_token]*batch_size],dtype=torch.long,device=device).transpose(0,1)
        decoder_hidden=encoder_hidden
        loss=0.0
        use_teacher_forcing=True if random.random()<teacher_forcing_ratio else False
        if use_teacher_forcing:
            for di in range(y.size(1)):
                decoder_output,decoder_hidden,decoder_attention=decoder(decoder_input,decoder_hidden,encoder_outputs,mask)
                loss+=criterion(decoder_output.squeeze(1),y[:,di])
                decoder_input=y[:,di].unsqueeze(1)
        else:
            for di in range(y.size(1)):
                decoder_output,decoder_hidden,decoder_attention=decoder(decoder_input,decoder_hidden,encoder_outputs,mask)
                topv,topi=decoder_output.topk(1)
                decoder_input=topi.squeeze(-1).detach()
                loss+=criterion(decoder_output.squeeze(1),y[:,di])
                if (topi==EOS_token).all():
                    break
        optimizer_encoder.zero_grad()
        optimizer_decoder.zero_grad()
        loss.backward()
        total_norm=torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(decoder.parameters()), max_norm=20.0)
        optimizer_encoder.step()
        optimizer_decoder.step()
        total_loss+=loss.item()/y.size(1)
        interval_loss+=loss.item()/y.size(1)
        if iter_num%interval==0:
            print(f'第 {epoch+1} 轮, 迭代次数: {iter_num}, 参数总范数: {total_norm:.4f}, 最近{interval}次迭代损失: {interval_loss/interval:.4f}, 总损失: {total_loss/iter_num:.4f}, 时间: {time.time()-start_time:.2f}s')
            interval_loss=0.0
    print(f'第 {epoch+1} 轮, 总损失: {total_loss/iter_num:.4f}, 此轮总时间: {time.time()-start_time:.2f}s')

torch.save(encoder.state_dict(), '../models/encoder.pt')
torch.save(decoder.state_dict(), '../models/decoder.pt')